<div class="alert alert-info">

## <center> CS587 Final Project – Phase 1 (Waterfall)</center>
### <center> GenAI-Assisted Project Planning for a Personalized Learning Management System (LMS) </center>

<br>

**Framework:** LangChain + Anthropic Claude (Sonnet 4)  
**Experiment:** Multi-Agent Simulation for Waterfall SDLC Planning  

<br>

**AI Agents:**
1. Customer Proxy
2. Project Manager
3. Requirements Engineer
4. System Engineer
5. Software Engineer
6. Test Engineer
7. Documentation Engineer

<br>

**Next Speaker Selection:** Round Robin order

</div>

In [12]:
# Install LangChain and Anthropic integration
import subprocess, sys
subprocess.check_call([sys.executable, "-m", "pip", "install",
                       "langchain-anthropic", "langchain-core", "--quiet"])


[notice] A new release of pip is available: 23.0.1 -> 26.0.1
[notice] To update, run: pip install --upgrade pip


0

In [13]:
import os
import json
import time
from datetime import datetime
from langchain_anthropic import ChatAnthropic
from langchain_core.messages import HumanMessage, SystemMessage

In [14]:
# Load API key from .env file
env_path = os.path.join(os.getcwd(), ".env")
if os.path.exists(env_path):
    with open(env_path) as f:
        for line in f:
            line = line.strip()
            if line and not line.startswith("#") and "=" in line:
                key, value = line.split("=", 1)
                os.environ[key.strip()] = value.strip()

ANTHROPIC_API_KEY = os.environ.get("ANTHROPIC_API_KEY")
if not ANTHROPIC_API_KEY:
    raise ValueError("ANTHROPIC_API_KEY not found. Create a .env file with: ANTHROPIC_API_KEY=sk-ant-...")

MODEL = "claude-sonnet-4-20250514"

llm = ChatAnthropic(
    model=MODEL,
    anthropic_api_key=ANTHROPIC_API_KEY,
    max_tokens=4096,
    temperature=0.7,
)

print(f"API Key loaded successfully. Model: {MODEL}")

API Key loaded successfully. Model: claude-sonnet-4-20250514


In [15]:
# ANSI escape codes (matching AutoGen style)
YELLOW = "\033[33m"
RED    = "\033[31m"
RESET  = "\033[0m"

SEPARATOR = "-" * 80

## Define AI Agents

Each agent has a system prompt that defines its role and responsibilities in the Waterfall SDLC.

In [16]:
AGENT_SYSTEM_PROMPTS = {
    "Requirement_Engineer": """You are a Requirements Engineer AI Agent for a software project using the Waterfall development model.

Your responsibilities:
1. Create a COMPLETE use case list with unique IDs (UC-001, UC-002, etc.) — provide at least 15-20 use cases
2. For each use case, provide: Use Case ID, Use Case Name, Primary Actor, Preconditions, Postconditions, Main Flow, and Alternate Flows
3. Create detailed FUNCTIONAL requirements (FR-001, FR-002, etc.) — at least 40+ requirements organized by category
4. Create NON-FUNCTIONAL requirements (NFR-001, NFR-002, etc.) — at least 15+ requirements

Then calculate the effort:
- step 1: work = total number of requirements documented
- step 2: productivity rate = 5 requirements completed every day
- step 3: effort = work / productivity

Provide everything in a well-structured markdown format with tables where appropriate.""",

    "System_Engineer": """You are a System Engineer AI Agent for a software project using the Waterfall development model.

Your responsibilities:
1. Create a HIGH-LEVEL SYSTEM ARCHITECTURE with clear layered design (Client, API/Orchestration, AI/Data Intelligence, Storage, Cross-Cutting)
2. Provide a COMPONENT DIAGRAM (textual description)
3. Provide a SEQUENCE DIAGRAM for a key use case (textual description with numbered steps)
4. Create an ENTITY-RELATIONSHIP DIAGRAM (textual) with all entities, attributes, and relationships
5. Describe the TECHNOLOGY STACK recommendations

Then calculate the effort:
- step 1: work = estimated total number of pages in the design document
- step 2: productivity rate = 5 pages completed every day
- step 3: effort = work / productivity

Provide everything in well-structured markdown format.""",

    "Software_Engineer": """You are a Software Developer AI Agent for a software project using the Waterfall development model.

Your responsibilities:
1. Create a detailed IMPLEMENTATION TASK LIST with task IDs (T1, T2, T3...)
2. For each task, provide: description, estimated hours, and dependencies
3. Group tasks by category (Foundation & Infrastructure, Core Backend, Core Frontend, GenAI Integration, Personalization, Admin & Analytics, Quality/Security/Docs)
4. Provide a DEPENDENCY OVERVIEW showing the build order
5. Describe the DEVELOPMENT APPROACH in sequential phases

Then calculate the effort:
- step 1: work = estimated total number of source lines of code (SLOC)
- step 2: productivity rate = 50 SLOC completed every day
- step 3: effort = work / productivity

Provide everything in well-structured markdown format with tables.""",

    "Test_Engineer": """You are a Test Engineer AI Agent for a software project using the Waterfall development model.

Your responsibilities:
1. Define the overall TEST STRATEGY (testing levels: unit, integration, system, AI evaluation)
2. Create FUNCTIONAL TEST CASES (TC-01 through TC-10+) with: Related UCs/FRs, Preconditions, Steps, Expected Results
3. Create NON-FUNCTIONAL TEST CASES (NTC-01 through NTC-06+) covering: Performance, AI Latency, Availability, Security/RBAC, Usability, Data Privacy
4. Provide a TESTING SCHEDULE broken into weeks

Then calculate the effort:
- step 1: work = estimated total number of test cases
- step 2: productivity rate = 2 test cases completed every day
- step 3: effort = work / productivity

Provide everything in well-structured markdown format with tables.""",

    "Documentation_Engineer": """You are a Documentation Engineer AI Agent for a software project using the Waterfall development model.

Your responsibilities:
1. Create an END-USER DOCUMENTATION OUTLINE (for students, instructors, admins)
2. Create a DEVELOPER DOCUMENTATION OUTLINE (for engineers)
3. Provide a DOCUMENTATION SCHEDULE showing pages per day and weekly breakdown
4. Estimate total pages for user docs and developer docs

Then calculate the effort:
- step 1: work = estimated total number of documentation pages
- step 2: productivity rate = 3 pages completed every day
- step 3: effort = work / productivity

Provide everything in well-structured markdown format.""",
}

## Customer Problem Statement & Conversation Flow

In [17]:
customer_message = (
    "I want to build a GenAI-Assisted Personalized Learning Management System (LMS) "
    "with the following core objectives and capabilities:\n\n"
    "1. Personalized Learning Paths for Students: Use GenAI to analyze a student's "
    "goals, current knowledge, performance, and preferences. Automatically generate "
    "and continuously adapt personalized learning paths (modules, lessons, assessments, "
    "and projects). Recommend content, practice questions, and projects tailored to "
    "each student.\n\n"
    "2. AI Tutor for Students: Provide an AI tutor chatbot inside the LMS that can "
    "answer course-related questions in natural language, explain concepts at different "
    "difficulty levels (beginner to advanced), generate examples, quizzes, and hints, "
    "and suggest next steps or remedial content when students are stuck.\n\n"
    "3. AI Assistance for Instructors: Help instructors create course materials "
    "including lecture outlines, slide drafts, reading summaries, quiz questions, "
    "assignments, rubrics, and sample answers. Allow instructors to input a topic or "
    "syllabus and have the system propose structured modules, learning objectives, "
    "and suggested assessments.\n\n"
    "4. Progress Tracking Dashboards: Dashboards for students showing learning path "
    "progress, completed modules, scores, and identified weak areas with AI-generated "
    "insights and recommendations. Dashboards for instructors/admins with class-level "
    "analytics, engagement, completion rates, average performance by topic, and "
    "individual student insights including risk of falling behind.\n\n"
    "5. Automated Project Planning Support Using AI Agents: Include AI agents that "
    "help students and/or instructors plan projects by breaking down a project into "
    "phases, tasks, and milestones; suggesting timelines, deliverables, and checklists; "
    "and adjusting plans based on progress updates.\n\n"
    "6. General Expectations: The system should be designed as a modern web application, "
    "architected to integrate with one or more LLM/GenAI providers (e.g., OpenAI or "
    "local models), with security, privacy, and role-based access for students, "
    "instructors, and admins."
)

# Project Manager messages to each agent
pm_to_requirement_engineer = (
    'I have received the following message from a customer: "' + customer_message + '", '
    "based on this message generate a list of detailed use-cases and requirements "
    "to develop such an application. Give every use-case a unique name and number "
    "and give every requirement for every use-case a unique number as well. "
    "Provide me with estimates about the productivity of the requirements engineer. "
    "How many use-cases and requirements can the engineer author and document every "
    "day/hour. What are the different tasks needed for the requirements phase? "
    "Include reviews and rework tasks in the list of tasks created."
)

pm_to_system_engineer = (
    "The Requirement Engineer has completed documenting the requirements. "
    "Please proceed with creating the detailed design document which includes all "
    "design elements based on the requirements document created by the requirement "
    "engineer for the requested software project. "
    "Also provide detailed estimates for the amount of work, effort, and productivity "
    "required to complete the system design phase measured by the number of pages "
    "created for the design document. Once completed, report back to me so I can "
    "proceed with the next steps in the project plan."
)

pm_to_software_engineer = (
    "The System Engineer has completed the architecture design. "
    "Please proceed with writing complete source code step by step and develop "
    "the software based on each design element of the design document created by "
    "the system engineer and each requirement from the requirements document by "
    "the requirement engineer. Also provide detailed estimates for the amount of "
    "work, effort, and productivity required for the development phase measured "
    "by number of source lines of code (SLOC) written. Once completed, report back "
    "to me so we can proceed with the next steps in the project plan."
)

pm_to_test_engineer = (
    "The software has been developed. Please proceed with creating the detailed "
    "test plan document which has all test cases and execute the test cases based "
    "on the software written by the software engineer, design document created by "
    "the system engineer and requirements document created by the requirement "
    "engineer. Also provide detailed estimates for the amount of work, effort, "
    "and productivity required to complete the testing phase measured by the number "
    "of test cases executed. Once completed, report back to me so we can proceed "
    "with the next steps in the project plan."
)

pm_to_documentation_engineer = (
    "The testing is complete. Please proceed with creating the detailed documentation "
    "which includes user documentation and training material based on the software "
    "written by the software engineer, design document created by the system engineer, "
    "and requirements document created by the requirement engineer. Also provide "
    "detailed estimates for the amount of work, effort, and productivity required for "
    "the documentation phase, measured by the number of pages created. Once completed, "
    "report back to me so we can finalize the project."
)

# Agent handoff responses
re_response = (
    "I have documented the use cases and numbered each requirement as requested. "
    "I have provided the work and effort estimates along with the productivity rate "
    "specified for the requirement engineering phase. "
    "I am now handing this over back to you to proceed with the next steps in the project plan."
)
se_response = (
    "I have completed the design document based on the provided requirements document. "
    "I have provided the work and effort estimates along with the productivity rate "
    "specified for the design phase. "
    "I am now handing this over back to you to proceed with the next steps in the project plan."
)
swe_response = (
    "I have completed the software development based on the design document and "
    "requirements document. I have provided the work and effort estimates along "
    "with the productivity rate specified for the implementation phase. "
    "I am now handing this over back to you to proceed with the next steps in the project plan."
)
te_response = (
    "I have completed the testing phase and verified that the software meets the requirements. "
    "I have provided the work and effort estimates along with the productivity rate "
    "specified for the testing phase. "
    "I am now handing this over back to you to proceed with the next steps in the project plan."
)
de_response = (
    "I have completed the documentation, including user documentation and training material. "
    "I have provided the work and effort estimates along with the productivity rate "
    "specified for the documentation phase. "
    "I am now handing this over back to you to proceed with finalizing the project."
)

In [18]:
# Conversation flow (round-robin like AutoGen GroupChat)
conversation_flow = [
    ("Customer_Proxy", "Project_Manager", customer_message),
    ("chat_manager", "Requirement_Engineer", pm_to_requirement_engineer),
    ("Requirement_Engineer", "chat_manager", re_response),
    ("chat_manager", "System_Engineer", pm_to_system_engineer),
    ("System_Engineer", "chat_manager", se_response),
    ("chat_manager", "Software_Engineer", pm_to_software_engineer),
    ("Software_Engineer", "chat_manager", swe_response),
    ("chat_manager", "Test_Engineer", pm_to_test_engineer),
    ("Test_Engineer", "chat_manager", te_response),
    ("chat_manager", "Documentation_Engineer", pm_to_documentation_engineer),
    ("Documentation_Engineer", "chat_manager", de_response),
]

## Helper Functions

In [19]:
def call_agent(agent_name, system_prompt, user_message, conversation_history=None):
    """Call an AI agent via LangChain (Claude Sonnet 4) and return its response."""
    messages = [SystemMessage(content=system_prompt)]

    if conversation_history:
        for entry in conversation_history[-3:]:
            messages.append(HumanMessage(
                content=f"[Previous output from {entry['agent']}]:\n{entry['response'][:3000]}"
            ))

    messages.append(HumanMessage(content=user_message))

    start_time = time.time()

    try:
        response = llm.invoke(messages)
        elapsed = time.time() - start_time

        # Extract token usage from LangChain/Anthropic response metadata
        tokens_used = 0
        if hasattr(response, 'usage_metadata') and response.usage_metadata:
            tokens_used = response.usage_metadata.get('total_tokens', 0)
        elif hasattr(response, 'response_metadata'):
            usage = response.response_metadata.get('usage', {})
            tokens_used = usage.get('input_tokens', 0) + usage.get('output_tokens', 0)

        return {
            "agent": agent_name,
            "response": response.content,
            "tokens": tokens_used,
            "time_seconds": round(elapsed, 2),
            "model": MODEL,
        }

    except Exception as e:
        print(f"Error from {agent_name}: {e}")
        return {
            "agent": agent_name,
            "response": f"ERROR: {str(e)}",
            "tokens": 0,
            "time_seconds": 0,
            "model": MODEL,
        }


def print_autogen_style(sender, receiver, message):
    """Print a message in AutoGen's group chat style."""
    print(f"{YELLOW}{sender}{RESET} (to {receiver}):\n")
    print(message)
    print(f"\n{SEPARATOR}")

## Run Experiment

Execute the full Waterfall project planning simulation with all AI agents.

In [20]:
all_results = []
conversation_history = []

for sender, receiver, message in conversation_flow:

    if receiver in AGENT_SYSTEM_PROMPTS:
        print_autogen_style(sender, receiver, message)
        print()

        print(f"{RED}")
        print(f">>>>>>>> NO HUMAN INPUT RECEIVED.")
        print(f">>>>>>>> USING AUTO REPLY...{RESET}")

        result = call_agent(
            agent_name=receiver,
            system_prompt=AGENT_SYSTEM_PROMPTS[receiver],
            user_message=message,
            conversation_history=conversation_history,
        )

        print_autogen_style(receiver, "chat_manager", result['response'])

        all_results.append(result)
        conversation_history.append(result)

        time.sleep(1)

    else:
        print_autogen_style(sender, receiver, message)
        print()

Customer_Proxy (to Project_Manager):

I want to build a GenAI-Assisted Personalized Learning Management System (LMS) with the following core objectives and capabilities:

1. Personalized Learning Paths for Students: Use GenAI to analyze a student's goals, current knowledge, performance, and preferences. Automatically generate and continuously adapt personalized learning paths (modules, lessons, assessments, and projects). Recommend content, practice questions, and projects tailored to each student.

2. AI Tutor for Students: Provide an AI tutor chatbot inside the LMS that can answer course-related questions in natural language, explain concepts at different difficulty levels (beginner to advanced), generate examples, quizzes, and hints, and suggest next steps or remedial content when students are stuck.

3. AI Assistance for Instructors: Help instructors create course materials including lecture outlines, slide drafts, reading summaries, quiz questions, assignments, rubrics, and sample

## Save Results

In [21]:
output_dir = "Phase1_Experiment_Results_Claude"
os.makedirs(output_dir, exist_ok=True)

for result in all_results:
    filepath = os.path.join(output_dir, f"{result['agent']}_output.md")
    with open(filepath, "w") as f:
        f.write(f"# {result['agent'].replace('_', ' ')} Output\n\n")
        f.write(f"**Model:** {result['model']}\n\n")
        f.write(f"**Tokens Used:** {result['tokens']}\n\n")
        f.write(f"**Response Time:** {result['time_seconds']}s\n\n")
        f.write(f"---\n\n")
        f.write(result['response'])
    print(f"Saved: {filepath}")

# Combined report
combined_path = os.path.join(output_dir, "Phase1_Waterfall_Complete_Report.md")
section_names = {
    "Requirement_Engineer": "SECTION 2 — REQUIREMENTS ENGINEER AI AGENT",
    "System_Engineer": "SECTION 3 — SYSTEM ENGINEER AI AGENT",
    "Software_Engineer": "SECTION 4 — SOFTWARE DEVELOPER AI AGENT",
    "Test_Engineer": "SECTION 5 — TEST ENGINEER AI AGENT",
    "Documentation_Engineer": "SECTION 6 — DOCUMENTATION ENGINEER AI AGENT",
}
with open(combined_path, "w") as f:
    f.write("# Phase 1 – Waterfall Project Plan\n")
    f.write("# GenAI-Assisted Personalized Learning Management System (LMS)\n\n")
    f.write(f"**Generated:** {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}\n\n")
    f.write(f"**LLM Model:** {MODEL}\n\n")
    f.write(f"**Framework:** LangChain + Anthropic Claude (Multi-Agent Simulation)\n\n")
    f.write("---\n\n")
    f.write("## SECTION 1 — CUSTOMER PROXY & PROJECT MANAGER\n\n")
    f.write("### 1.1 Customer Problem Statement\n\n")
    f.write(customer_message.strip())
    f.write("\n\n---\n\n")
    for result in all_results:
        agent = result['agent']
        if agent in section_names:
            f.write(f"## {section_names[agent]}\n\n")
            f.write(f"**Model:** {result['model']} | **Tokens:** {result['tokens']} | **Time:** {result['time_seconds']}s\n\n")
            f.write(result['response'])
            f.write("\n\n---\n\n")
print(f"Saved combined report: {combined_path}")

# Metadata
metadata = {
    "experiment": {
        "title": "Phase 1 – Waterfall Project Planning",
        "project": "GenAI-Assisted Personalized LMS",
        "model": MODEL,
        "framework": "LangChain + Anthropic Claude (Multi-Agent Simulation)",
        "date": datetime.now().isoformat(),
        "total_tokens": sum(r['tokens'] for r in all_results),
        "total_time_seconds": sum(r['time_seconds'] for r in all_results),
    },
    "agents": [{"name": r['agent'], "tokens": r['tokens'], "time_seconds": r['time_seconds'], "model": r['model']} for r in all_results]
}
metadata_path = os.path.join(output_dir, "experiment_metadata.json")
with open(metadata_path, "w") as f:
    json.dump(metadata, f, indent=2)
print(f"Saved metadata: {metadata_path}")

Saved: Phase1_Experiment_Results_Claude/Requirement_Engineer_output.md
Saved: Phase1_Experiment_Results_Claude/System_Engineer_output.md
Saved: Phase1_Experiment_Results_Claude/Software_Engineer_output.md
Saved: Phase1_Experiment_Results_Claude/Test_Engineer_output.md
Saved: Phase1_Experiment_Results_Claude/Documentation_Engineer_output.md
Saved combined report: Phase1_Experiment_Results_Claude/Phase1_Waterfall_Complete_Report.md
Saved metadata: Phase1_Experiment_Results_Claude/experiment_metadata.json


## Experiment Summary

In [22]:
total_tokens = sum(r['tokens'] for r in all_results)
total_time = sum(r['time_seconds'] for r in all_results)

print(f"{'=' * 60}")
print(f"  EXPERIMENT COMPLETE")
print(f"{'=' * 60}")
print()
print(f"  Model:         {MODEL}")
print(f"  Framework:     LangChain + Anthropic Claude")
print(f"  Total Agents:  {len(all_results)}")
print(f"  Total Tokens:  {total_tokens:,}")
print(f"  Total Time:    {total_time:.1f}s")
print()
print(f"  Agent Breakdown:")
for r in all_results:
    print(f"    {r['agent']:30s} | Tokens: {r['tokens']:6,} | Time: {r['time_seconds']:.1f}s")
print()
print(f"{'=' * 60}")

  EXPERIMENT COMPLETE

  Model:         claude-sonnet-4-20250514
  Framework:     LangChain + Anthropic Claude
  Total Agents:  5
  Total Tokens:  26,715
  Total Time:    279.6s

  Agent Breakdown:
    Requirement_Engineer           | Tokens:  4,867 | Time: 64.2s
    System_Engineer                | Tokens:  4,692 | Time: 57.4s
    Software_Engineer              | Tokens:  5,070 | Time: 50.2s
    Test_Engineer                  | Tokens:  6,721 | Time: 64.2s
    Documentation_Engineer         | Tokens:  5,365 | Time: 43.6s

